# Optimizers in Detail: Gradient Descent, Momentum, and Adam
Accompanies **Unit 2.4: Optimizers in Detail**.

When closed-form normal equations are unavailable or computationally intractable, we use numerical optimization.
In this notebook, we trace the geometric mechanics of:
1. Standard Batch Gradient Descent
2. Stochastic Gradient Descent (SGD) with Mini-Batching
3. Polyak's Momentum (damping oscillations along steep canyon walls)
4. Adam (Adaptive Moment Estimation with analytical bias correction)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Define an anisotropic quadratic loss surface: f(w1, w2) = 10 * w1^2 + w2^2
# Curvature along w1 is 10x steeper than along w2 (condition number kappa = 10)
def loss_func(w1, w2):
    return 10.0 * (w1 ** 2) + (w2 ** 2)

def grad_func(w1, w2):
    return np.array([20.0 * w1, 2.0 * w2])

w_start = np.array([2.0, 2.0])

### Standard Gradient Descent: Anisotropic Ravine Oscillation
With a fixed learning rate $\eta = 0.08$, gradient descent bounces transversely across the steep walls of $w_1$ while crawling slowly along $w_2$.

In [ ]:
def run_gd(w_init, lr=0.08, steps=25):
    trajectory = [w_init.copy()]
    w = w_init.copy()
    for _ in range(steps):
        g = grad_func(w[0], w[1])
        w -= lr * g
        trajectory.append(w.copy())
    return np.array(trajectory)

traj_gd = run_gd(w_start, lr=0.08, steps=25)
print(f"GD final position: ({traj_gd[-1, 0]:.4f}, {traj_gd[-1, 1]:.4f}) | Cost: {loss_func(traj_gd[-1, 0], traj_gd[-1, 1]):.4f}")

### Polyak's Heavy-Ball Momentum
Momentum accumulates an exponentially decaying velocity vector:
$$\mathbf{v}_t = \beta \mathbf{v}_{t-1} + \eta \nabla C(\mathbf{w}_t), \quad \mathbf{w}_{t+1} = \mathbf{w}_t - \mathbf{v}_t$$
Opposing transverse oscillations cancel out while longitudinal progress accelerates.

In [ ]:
def run_momentum(w_init, lr=0.02, beta=0.85, steps=25):
    trajectory = [w_init.copy()]
    w = w_init.copy()
    v = np.zeros_like(w)
    for _ in range(steps):
        g = grad_func(w[0], w[1])
        v = beta * v + lr * g
        w -= v
        trajectory.append(w.copy())
    return np.array(trajectory)

traj_mom = run_momentum(w_start, lr=0.02, beta=0.85, steps=25)
print(f"Momentum final position: ({traj_mom[-1, 0]:.4f}, {traj_mom[-1, 1]:.4f}) | Cost: {loss_func(traj_mom[-1, 0], traj_mom[-1, 1]):.4f}")

### Adam: Adaptive Moment Estimation
Adam computes first moments (momentum) and second uncentered moments (RMSprop) with bias correction:
$$\hat{\mathbf{m}}_t = \frac{\mathbf{m}_t}{1 - \beta_1^t}, \quad \hat{\mathbf{v}}_t = \frac{\mathbf{v}_t}{1 - \beta_2^t}, \quad \mathbf{w}_{t+1} = \mathbf{w}_t - \frac{\eta}{\sqrt{\hat{\mathbf{v}}_t} + \epsilon} \hat{\mathbf{m}}_t$$

In [ ]:
def run_adam(w_init, lr=0.4, beta1=0.9, beta2=0.999, eps=1e-8, steps=25):
    trajectory = [w_init.copy()]
    w = w_init.copy()
    m = np.zeros_like(w)
    v = np.zeros_like(w)
    for t in range(1, steps + 1):
        g = grad_func(w[0], w[1])
        m = beta1 * m + (1.0 - beta1) * g
        v = beta2 * v + (1.0 - beta2) * (g ** 2)
        m_hat = m / (1.0 - beta1 ** t)
        v_hat = v / (1.0 - beta2 ** t)
        w -= (lr / (np.sqrt(v_hat) + eps)) * m_hat
        trajectory.append(w.copy())
    return np.array(trajectory)

traj_adam = run_adam(w_start, lr=0.4, steps=25)
print(f"Adam final position: ({traj_adam[-1, 0]:.4f}, {traj_adam[-1, 1]:.4f}) | Cost: {loss_func(traj_adam[-1, 0], traj_adam[-1, 1]):.4f}")

### Visualizing Optimization Trajectories Across Contour Ravine

In [ ]:
w1_grid = np.linspace(-2.5, 2.5, 200)
w2_grid = np.linspace(-2.5, 2.5, 200)
W1, W2 = np.meshgrid(w1_grid, w2_grid)
Z = loss_func(W1, W2)

plt.figure(figsize=(9, 6))
plt.contour(W1, W2, Z, levels=[0.5, 2, 5, 12, 25, 45], cmap='Blues_r', linewidths=1.2)
plt.plot(traj_gd[:, 0], traj_gd[:, 1], 'ro-', label='Standard GD (Oscillating)', markersize=4)
plt.plot(traj_mom[:, 0], traj_mom[:, 1], 'go-', label='Momentum (Damped)', markersize=4)
plt.plot(traj_adam[:, 0], traj_adam[:, 1], 'bo-', label='Adam (Rescaled Axis)', markersize=4)
plt.plot(0, 0, 'k*', markersize=12, label='Global Minimum (0,0)')
plt.xlabel('$w_1$ (Steep axis)', fontsize=11)
plt.ylabel('$w_2$ (Flat axis)', fontsize=11)
plt.title('Optimization Trajectories on Anisotropic Quadratic Basin', fontsize=12, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.show()